# 05 · Cost & Latency

> **Source notes:** `CostAndLatency.md`

This notebook makes the cost & latency formulas concrete:
- **Token counting & cost estimation** — tiktoken + pricing calculator
- **Context composition** — where tokens come from in a RAG pipeline
- **Conversation history leak** — quantifying the biggest cost driver
- **Prefix caching savings** — realistic calculation
- **Latency decomposition** — TTFT, generation, post-processing budgets

No LLM API key needed — all calculations use local token counting.

In [ ]:
def count_tokens(text):
    """
    TODO #1: Implement `count_tokens()`.

    Steps:
    1. Define helper function `count_tokens()`

    Hint:
    enc = tiktoken.get_encoding(???)
    enc.encode(???)

    Returns: result
    """
    raise NotImplementedError("TODO: implement count_tokens()")

## 1 · Cost Formula

```
Cost = (input_tokens x input_$/1M) + (output_tokens x output_$/1M)
```
Every architectural decision maps back to this formula.

In [ ]:
def cost_per_call(inp, outp, model):
    """
    TODO #2: Implement `cost_per_call()`.

    Steps:
    1. Define helper function `cost_per_call()`
    2. Compute `system_prompt` using `99()`
    3. Compute `INPUT_PARTS` using `items()`

    Hint:
    # implement using the APIs described above

    Returns: (inp * p['input'] + outp * p['output'...
    """
    raise NotImplementedError("TODO: implement cost_per_call()")

## 2 · Conversation History — Biggest Cost Leak

Passing full conversation history on every call creates **linearly growing** costs per session.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `BASE_TOKENS` using `count_tokens()`
# 2. Compute `history` using `count_tokens()`
#
# Hint:
#    # implement using the APIs described above

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
BASE_TOKENS   = count_tokens(system_prompt + user_message)
MODEL_        = 'gpt-4o-mini'
OUTPUT_TOK    = 50
TURNS = [
    ('What toppings are on Pepperoni?', 'Pepperoni, mozzarella, tomato sauce.'),
    ('Is there a gluten-free option?',  'Yes, GF base is +1.50.'),
    ('What is the minimum order?',      'Minimum order is 15.'),
    ('Do you deliver north zone Mon?',  'North zone Mon-Fri only.'),
    ('What time Sunday close?',         'Sunday 9pm.'),
] * 2

history = ''
session_cost = 0.0
print(f'{"Turn":>5} {"Input tokens":>14} {"Cost/call":>12} {"Session cost":>13}')
print('-'*50)
for i, (q, a) in enumerate(TURNS[:8], 1):
    input_tokens  = BASE_TOKENS + count_tokens(history)
    c             = cost_per_call(input_tokens, OUTPUT_TOK, MODEL_)
    session_cost += c
    print(f'{i:>5} {input_tokens:>14} {c:>12.6f} {session_cost:>13.6f}')
    history += f'User: {q}\nBot: {a}\n'
print(f'\nSession cost 8 turns (mid): ${session_cost:.4f}')
print(f'At 1,000 sessions/day: ${session_cost*1000:.2f}/day')

## 3 · Prefix Caching Savings

Static prefix (system prompt + few-shot) = 90% discounted when it remains identical across calls.
Put static content **first**, keep it **identical** across calls.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `STATIC` using `count_tokens()`
#
# Hint:
#    # implement using the APIs described above

<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
STATIC   = system_prompt + '\n\n' + few_shot
DYNAMIC  = retrieved_chunks + '\n' + user_message
stat_tok = count_tokens(STATIC)
dyn_tok  = count_tokens(DYNAMIC)
tot_tok  = stat_tok + dyn_tok
CALLS    = 10_000
DISC     = 0.90
outp     = count_tokens(typical_output)
cost_nc  = cost_per_call(tot_tok, outp, 'gpt-4o-mini') * CALLS
eff_inp  = dyn_tok + stat_tok * (1 - DISC)
cost_c   = cost_per_call(eff_inp, outp, 'gpt-4o-mini') * CALLS
savings  = cost_nc - cost_c
print(f'Static prefix tokens : {stat_tok:>6}  ({stat_tok/tot_tok:.0%} of total)')
print(f'Dynamic tokens/call  : {dyn_tok:>6}')
print(f'Daily cost no cache  : ${cost_nc:>8.2f}')
print(f'Daily cost cached    : ${cost_c:>8.2f}')
print(f'Daily savings        : ${savings:>8.2f}  ({savings/cost_nc:.1%})')
print(f'Annual savings       : ${savings*365:>8.2f}')

## 4 · Latency Decomposition

```
Total latency = network_RTT + TTFT + (output_tokens x ms/token) + [optional post-processing]
```
TTFT scales with input token count. ms/token scales with model size.

In [ ]:
def est_lat(inp, outp, nli=False, sc=False):
    """
    TODO #5: Implement `est_lat()`.

    Steps:
    1. Compute `P`
    2. Define helper function `est_lat()`
    3. Compute `configs` using `chat()`

    Hint:
    # implement using the APIs described above

    Returns: round(ttft), round(gen), round(tot/10...
    """
    raise NotImplementedError("TODO: implement est_lat()")

## Summary

| Lever | Impact | Action |
|---|---|---|
| Model tier | 10-100x cost | Use cheapest passing eval |
| Context length | Biggest driver | Summarise history; short chunks |
| Prefix caching | 40-80% reduction | Static content first, identical across calls |
| Self-consistency | 5x latency | Reserve for high-stakes only |
| Streaming | Perceived latency | Always stream user-visible text |

**Next:** [EvaluatingAISystems/notebook.ipynb](../EvaluatingAISystems/notebook.ipynb)